In [ ]:
# Installing required verions of sympy
!pip uninstall -y sympy
!pip install sympy==1.13.1

In [ ]:
import torch
from transformers import AutoTokenizer, AutoModel
import pandas as pd
import numpy as np

In [ ]:
device = "cuda"
model_name = "facebook/esm2_t30_150M_UR50D"               #ESM2 150M  my meta -- Proteain language Model
tokenizer = AutoTokenizer.from_pretrained(model_name)
model = AutoModel.from_pretrained(model_name).to(device)
model.eval()
print('Model Sucessfully Loaded')

In [ ]:
proteases = pd.read_csv('/content/uniprotkb_ec_3_4_21_AND_length_450_TO_5_2026_04_14 (1).tsv',sep = '\t') # Loading data
embedding_vec = []          # List to store embedding
for i in range(len(proteases)):
  sequence = proteases.iloc[i]['Sequence']
  inputs = tokenizer(sequence, return_tensors="pt").to(device)
  with torch.no_grad():
      outputs = model(**inputs)
  embeddings = outputs.last_hidden_state[:, 1:-1] # Extracting Last layer embdding. CLS token can also be used
  sequence_embedding = embeddings.mean(dim=1)     # Averaging it according to sequence size
  sequence_embedding = sequence_embedding.cpu().detach().numpy()    # Converting to procesable form
  embedding_vec.append(sequence_embedding)
proteases['Encoding'] = embedding_vec   # Storing the computed encoding in the list


In [ ]:
hip_1 = proteases.iloc[21]['Encoding'][0]   # Taking the embeddings of Hip !
sim = []      # List to store the similarity score
for i in range(len(proteases)):
  enc = proteases.iloc[i]['Encoding'][0]
  cos_sim = np.dot(hip_1,enc)/(np.linalg.norm(hip_1)*np.linalg.norm(enc))   # Computing cosine similarity
  sim.append(cos_sim)
proteases['Similarity'] = sim

In [ ]:
sorted = proteases.sort_values(by = 'Similarity',ascending = False)
sorted = sorted[['Entry', 'Entry Name', 'Protein names',
       'Organism', 'Length', 'Mass', 'Active site'
      , 'Protein families', 'Encoding',
       'Similarity']]


In [ ]:
sorted